[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/longas14/Fisica_Computacional_1/blob/main/notebooks/numeros_aleatorios.ipynb)

---
# Física Computacional 1
Robinson Longas Bedoya  
*Facultad de Ciencias Exactas y Aplicadas - Institución Universitaria ITM*  
Contacto: [robinsonlongas@itm.edu.co](mailto:robinsonlongas@itm.edu.co) | [robinsonlongas3586@correo.itm.edu.co](mailto:robinsonlongas3586@correo.itm.edu.co)

---
## 🎲 Números Pseudoaleatorios, Caminatas Aleatorias y Geometría Fractal

### Bibliografía recomendada
- *Computational Physics*, Mark Newman.
- *Programming for Computations - Python*, Svein Linge and Petter Langtangen.
- *Computational Physics with Python*, Eric Ayars.
- *Computational Mathematics: An Introduction to Numerical Analysis and Scientific Computing with Python*, Dimitrios Mitsotakis.
- *Computational Physics: Problem Solving with Python*, Rubin H. Landau, Manuel J. Páez and Cristian C. Bordeianu.


---
## 1. Generador de Congruencia Lineal (LCG)

Si nos piden que digamos números al azar, comenzamos a nombrar los primeros que se nos vengan a la cabeza: tal vez los que más nos gusten, tal vez solo positivos, tal vez números primos... ¿En realidad son al azar? Supongamos que mejor se los pedimos al computador. ¿Resolvería esto el problema?

Recordemos que el computador opera de forma estrictamente determinista y, en rigor, **no puede generar azar verdadero**. Decimos que una secuencia de números $x_0, x_1, \dots, x_n$ es aleatoria si no existe una correlación estadística entre ellos. En arquitecturas deterministas generamos números **pseudoaleatorios**.

El método clásico por excelencia es el **Generador de Congruencia Lineal (GCL)**: fijamos una semilla $x_0$ y generamos los términos sucesivos mediante la relación de recurrencia:

$$
x_{n+1} = (a x_n + c) \pmod m
$$

> **Ejemplo:** Si tomamos $a=5$, $c=3$ y módulo $m=8$, con semilla inicial $x_0=1$, la secuencia es:
> $$0, 3, 2, 5, 4, 7, 6, 1, 0, 3, \dots$$
> Recorre todos los residuos módulo 8 antes de repetirse, completando un período $P = 8$.

| Sistema / Implementación | Tipo | $a$ | $c$ | $m$ |
| :--- | :---: | :---: | :---: | :---: |
| **IBM RANDU (System/360)** | LCG | $65\,539$ | $0$ | $2^{31}$ |
| **ANSI C (`rand` MSVC)** | LCG | $214\,013$ | $2\,531\,011$ | $2^{32}$ |
| **POSIX `drand48`** | LCG | $25\,214\,903\,917$ | $11$ | $2^{48}$ |
| **Numerical Recipes (32-bit)** | LCG | $1\,664\,525$ | $1\,013\,904\,223$ | $2^{32}$ |
| **NumPy (`default_rng`)** | PCG64 | $6\,364\,136\,223\,846\,793\,005$ | Dinámico (128-bit) | $2^{128}$ |

### El desastre de RANDU y el Teorema de Marsaglia
En IBM RANDU se escogió $a = 65539 = 2^{16} + 3$ para acelerar multiplicaciones mediante desplazamientos de bits (`x << 16`). En 1968, George Marsaglia demostró que ternas consecutivas $(x_n, x_{n+1}, x_{n+2})$ de RANDU **se concentran sobre apenas 15 planos bidimensionales en $\mathbb{R}^3$**, arruinando décadas de simulaciones nucleares y astrofísicas que debieron ser recalculadas.


In [ ]:
# numeros_aleatorios.py: Comparativa LCG manual vs PCG64 (NumPy)
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

def lcg(size, seed=23):
    """Genera un arreglo de N números en (0, 1) usando x_{n+1} = (a*x_n + c) mod m."""
    a = 1103515245
    c = 12345
    m = 2**31

    numeros = np.empty(size)
    xn = seed
    for i in range(size):
        xn = (a * xn + c) % m
        numeros[i] = xn / m
    return numeros

In [ ]:
N = 10000
semilla = 12

# 1. Generación de datos
datos_lcg = lcg(N, semilla) * 1000
rng = np.random.default_rng(seed=semilla)
datos_numpy = rng.random(N) * 1000

# 2. Gráficas de correlación (Espacio de fases x_n vs x_{n+1})
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(datos_lcg[:-1], datos_lcg[1:], s=1, color='crimson', alpha=0.5)
axes[0].set_title("Correlación $x_n$ vs $x_{n+1}$ (A mano - LCG)", fontweight='bold')
axes[0].set_xlabel("$x_n$")
axes[0].set_ylabel("$x_{n+1}$", labelpad=8)
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].scatter(datos_numpy[:-1], datos_numpy[1:], s=1, color='teal', alpha=0.5)
axes[1].set_title("Correlación $x_n$ vs $x_{n+1}$ (NumPy - PCG64)", fontweight='bold')
axes[1].set_xlabel("$x_n$")
axes[1].set_ylabel("$x_{n+1}$", labelpad=8)
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()

### Arquitectura de `numpy.random` (v1.17+)
NumPy separa el problema en dos niveles:
1. **`BitGenerator` (PCG64):** Rutina en C de ultra-alta velocidad que genera bits crudos pseudoaleatorios.
2. **`Generator` (`default_rng`):** Transforma analíticamente los bits en distribuciones estadísticas continuas o discretas.

| Método | Distribución / Operación | Aplicación física típica |
| :--- | :--- | :--- |
| `rng.random(size)` | Uniforme $U \in [0.0, 1.0)$ | Integración Monte Carlo, ruletas |
| `rng.uniform(low, high, size)` | Uniforme continua $[a, b)$ | Distribución espacial homogénea |
| `10**rng.uniform(a, b, size)` | Escala logarítmica $[10^a, 10^b]$ | Barridos de masas o acoplamientos |
| `rng.integers(low, high, size)` | Enteros equiprobables $[a, b)$ | Espines en Ising, caminos discretos |
| `rng.normal(loc, scale, size)` | Gaussiana $\mathcal{N}(\mu, \sigma)$ | Velocidades térmicas Maxwell-Boltzmann |
| `rng.exponential(scale, size)` | Exponencial $\lambda e^{-\lambda t}$ | Decaimiento radiactivo, camino libre medio |
| `rng.poisson(lam, size)` | Conteo de Poisson | Detección de fotones, eventos raros |


---
## EJempo: estimación de $\pi$ 

Inscribimos un círculo de radio $R=1$ centrado en el origen dentro de un cuadrado de lado $L = 2R = 2$, delimitado por $[-1, 1] \times [-1, 1]$:

* Área del cuadrado: $A_{\square} = L^2 = 4$
* Área del círculo: $A_{\bigcirc} = \pi R^2 = \pi$

Si disparamos $N$ puntos aleatorios uniformes sobre el cuadrado y contamos cuántos $M$ caen dentro del disco ($x^2 + y^2 \le 1$):

$$
\frac{A_{\bigcirc}}{A_{\square}} = \frac{M}{N} \implies \frac{\pi}{4} \approx \frac{M}{N} \implies \pi \approx 4\,\frac{M}{N}
$$

In [ ]:
# pi_random.py
def monte_carlo_pi(N=5000, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1, 1, N)
    y = rng.uniform(-1, 1, N)
    adentro = (x**2 + y**2) <= 1.0
    M = np.sum(adentro)
    pi_ran = 4.0 * M / N
    return x, y, adentro, pi_ran, M

In [ ]:
N = 5000
x, y, adentro, pi_aprox, M = monte_carlo_pi(N)

plt.figure(figsize=(6, 6), facecolor='white')
plt.scatter(x[adentro], y[adentro], s=3, color="crimson", label="Adentro")
plt.scatter(x[~adentro], y[~adentro], s=3, color="navy", alpha=0.3, label="Afuera")

# Círculo analítico
theta = np.linspace(0, 2 * np.pi, 300)
plt.plot(np.cos(theta), np.sin(theta), "k-", lw=2)

plt.axis("equal")
plt.xlim(-1.05, 1.05)
plt.ylim(-1.05, 1.05)
plt.title(f"Monte Carlo: $\\pi \\approx {pi_aprox:.4f}$ ($N = {N}$, $M = {M}$)", fontsize=12, fontweight="bold")
plt.xlabel("x", fontsize=11)
plt.ylabel("y", fontsize=11)
plt.legend(loc="upper right")
plt.tight_layout()

---
## Ejemplo: caminata Aleatoria 

Consideremos un caminante en 1D que parte de $x_0 = 0$. En cada paso de tiempo $\Delta t = 1$, avanza una variable discreta $s_i \in \{-1, +1\}$ con probabilidades idénticas $P(+1) = P(-1) = 1/2$.

* **Valor esperado (Desplazamiento medio):** $\langle X_N \rangle = \sum_{i=1}^N \langle s_i \rangle = 0$.
* **Desplazamiento Cuadrático Medio (MSD):** Como los pasos son estadísticamente independientes ($\langle s_i s_j \rangle = 0$ para $i \neq j$):

$$
\langle X_N^2 \rangle = \sum_{i=1}^N \langle s_i^2 \rangle + \sum_{i \neq j} \langle s_i s_j \rangle = N \cdot 1 + 0 = N
$$

> **Ley de difusión:** La distancia típica recorrida escala con la raíz cuadrada del número de pasos:
> $$
> x_{\text{rms}} = \sqrt{\langle X_N^2 \rangle} = \sqrt{N} \propto \sqrt{t}
> $$
> En contraste con el transporte balístico determinista donde la distancia crece proporcional a $t$.

In [ ]:
# drunk_walk.py: Simulación en 1D, 2D y 3D
import random

def drunk_1d(N, L=1.0):
    x = 0.0
    x_history = [x]
    for _ in range(N):
        x += (random.random() - 0.5) * 2.0 * L
        x_history.append(x)
    return x_history

def drunk_2d(N, L=1.0):
    x, y = 0.0, 0.0
    x_history, y_history = [x], [y]
    for _ in range(N):
        x += (random.random() - 0.5) * 2.0 * L
        y += (random.random() - 0.5) * 2.0 * L
        x_history.append(x)
        y_history.append(y)
    return x_history, y_history

def drunk_3d(N, L=1.0):
    x, y, z = 0.0, 0.0, 0.0
    x_history, y_history, z_history = [x], [y], [z]
    for _ in range(N):
        x += (random.random() - 0.5) * 2.0 * L
        y += (random.random() - 0.5) * 2.0 * L
        z += (random.random() - 0.5) * 2.0 * L
        x_history.append(x)
        y_history.append(y)
        z_history.append(z)
    return x_history, y_history, z_history

In [ ]:

N = 1000
L = 10.0
n_pasos = list(range(N + 1))
random.seed(42)

x_1d = drunk_1d(N, L=L)
x_2d, y_2d = drunk_2d(N, L=L)
x_3d, y_3d, z_3d = drunk_3d(N, L=L)

fig = plt.figure(figsize=(16, 4.5))

# Subplot 1: 1D
ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(n_pasos, x_1d, color='royalblue', lw=1.5, label='Trayectoria $x(N)$')
ax1.plot(0, x_1d[0], 'go', markersize=7, label='Inicio')
ax1.plot(N, x_1d[-1], 'ro', markersize=7, label='Fin')
ax1.set_title('Random Walk 1D', fontweight='bold')
ax1.set_xlabel('Pasos ($N$)')
ax1.set_ylabel('Posición $x$')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(fontsize=9)

# Subplot 2: 2D
ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(x_2d, y_2d, color='royalblue', lw=1.2, alpha=0.85)
ax2.plot(0, 0, 'go', markersize=7, label='Inicio (0,0)')
ax2.plot(x_2d[-1], y_2d[-1], 'ro', markersize=7, label='Fin')
ax2.set_title('Random Walk 2D', fontweight='bold')
ax2.set_xlabel('Posición $x$')
ax2.set_ylabel('Posición $y$')
ax2.set_aspect('equal', 'datalim')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(fontsize=9)

# Subplot 3: 3D
ax3 = fig.add_subplot(1, 3, 3, projection='3d')
ax3.plot(x_3d, y_3d, z_3d, color='royalblue', lw=1.2, alpha=0.85)
ax3.plot([0], [0], [0], 'go', markersize=7, label='Inicio (0,0,0)')
ax3.plot([x_3d[-1]], [y_3d[-1]], [z_3d[-1]], 'ro', markersize=7, label='Fin')
ax3.set_title('Random Walk 3D', fontweight='bold')
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_zlabel('z')
ax3.legend(fontsize=8)

plt.tight_layout()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
from IPython.display import HTML  # <--- Crucial para ver la animación en Jupyter

def drunk_nd(N, dim=1, L=1.0):
    """
    Simula la caminata en 1D, 2D o 3D con el algoritmo vectorizado.
    """
    if dim not in (1, 2, 3):
        raise ValueError("El parámetro 'dim' debe ser 1, 2 o 3.")
    
    # Genera la matriz de pasos aleatorios continuos Δk ~ [-L, L] de forma (N, dim)
    pasos = np.random.uniform(-L, L, size=(N, dim))
    
    # Posición inicial en el origen: (1, dim) de ceros
    origen = np.zeros((1, dim))
    
    # Concatena el origen y calcula la suma acumulada a lo largo del tiempo
    trayectoria = np.vstack([origen, np.cumsum(pasos, axis=0)])
    
    if dim == 1:
        return trayectoria.ravel()
        
    return trayectoria


def ejecutar_simulacion(DIM=2, N=150, L=1.0, guardar_gif=False):
    """
    Ejecuta y anima la simulación de Random Walk en 2D o 3D dentro del Notebook.
    """
    np.random.seed(42)  # Semilla para reproducibilidad

    traj = drunk_nd(N, dim=DIM, L=L)
    x, y = traj[:, 0], traj[:, 1]
    if DIM == 3:
        z = traj[:, 2]

    # Volumen del plot con margen
    margin = 2.0
    xlim = (np.min(x) - margin, np.max(x) + margin)
    ylim = (np.min(y) - margin, np.max(y) + margin)

    # Configuración de figura
    fig = plt.figure(figsize=(8, 6))

    if DIM == 3:
        zlim = (np.min(z) - margin, np.max(z) + margin)
        ax = fig.add_subplot(111, projection="3d")
        ax.set_zlim(zlim)
        ax.set_zlabel("Posición $z$")

        (linea_trayectoria,) = ax.plot([], [], [], color="crimson", lw=1.5, alpha=0.85, label="Trayectoria")
        (cabeza_caminante,) = ax.plot([], [], [], "bo", markersize=6, label="Posición actual")
        ax.plot([0], [0], [0], "go", markersize=8, label="Inicio (0,0,0)")
        texto_paso = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, fontsize=11, fontweight="bold")
    else:
        ax = fig.add_subplot(111)
        ax.grid(True, linestyle="--", alpha=0.5)

        (linea_trayectoria,) = ax.plot([], [], color="crimson", lw=1.5, alpha=0.85, label="Trayectoria")
        (cabeza_caminante,) = ax.plot([], [], "bo", markersize=6, label="Posición actual")
        ax.plot([0], [0], "go", markersize=8, label="Inicio (0,0)")
        texto_paso = ax.text(0.02, 0.95, "", transform=ax.transAxes, fontsize=11, fontweight="bold")

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(f"Evolución Temporal de Random Walk {DIM}D", fontweight="bold", fontsize=12)
    ax.set_xlabel("Posición $x$")
    ax.set_ylabel("Posición $y$")
    ax.legend(loc="upper right", fontsize=9)

    # Funciones para la animación
    def init():
        linea_trayectoria.set_data([], [])
        cabeza_caminante.set_data([], [])
        if DIM == 3:
            linea_trayectoria.set_3d_properties([])
            cabeza_caminante.set_3d_properties([])
        texto_paso.set_text("")
        return linea_trayectoria, cabeza_caminante, texto_paso

    def update(frame):
        linea_trayectoria.set_data(x[: frame + 1], y[: frame + 1])
        cabeza_caminante.set_data([x[frame]], [y[frame]])

        if DIM == 3:
            linea_trayectoria.set_3d_properties(z[: frame + 1])
            cabeza_caminante.set_3d_properties([z[frame]])
            ax.view_init(elev=20, azim=frame * 0.5)

        texto_paso.set_text(f"Paso: {frame} / {N}")
        return linea_trayectoria, cabeza_caminante, texto_paso

    anim = FuncAnimation(
        fig,
        update,
        frames=len(x),
        init_func=init,
        interval=40,
        blit=(DIM == 2),
        repeat=False,
    )

    if guardar_gif:
        nombre_archivo = f"random_walk_{DIM}d.gif"
        anim.save(nombre_archivo, writer="pillow", fps=30, dpi=100)
        print(f"¡GIF guardado como '{nombre_archivo}'!")

    # Cerramos la figura base
    plt.close(fig)

    # Retornamos el reproductor HTML interactivo de Jupyter
    return HTML(anim.to_jshtml())


# --- EJECUCIÓN  ---
ejecutar_simulacion(DIM=3, N=150, L=2.0, guardar_gif=False)

---
## Ejemplo: fractales de Mandelbrot y Julia

Ambos fractales surgen de la recursión cuadrática:
$$
z_{n+1} = z_n^2 + c, \quad z, c \in \mathbb{C}
$$

* **Conjunto de Mandelbrot:** Fijamos $z_0 = 0$ y variamos $c = x + iy$ en cada celda del plano.
* **Conjuntos de Julia:** Fijamos una constante compleja $c$ y variamos la condición inicial $z_0 = x + iy$.

Si en algún paso $|z| > 2$, se garantiza analíticamente que la órbita diverge al infinito. El valor numérico registrado en la matriz es el conteo de cuántas iteraciones tardó el punto en escapar.

### Guía de `plt.imshow()` frente a `plt.plot()`
* **`plt.plot(x, y)`**: Traza líneas o puntos 1D a partir de dos listas de coordenadas.
* **`plt.imshow(M)`**: Recibe una **matriz 2D** y colorea cada celda $(i, j)$ según su intensidad numérica.
* **`extent=[xmin, xmax, ymin, ymax]`**: Sustituye los índices discretos de píxeles ($0, 1, \dots, 1000$) por las coordenadas reales del espacio físico o complejo.
* **`origin="lower"`**: Fija $y_{\min}$ en la base del gráfico cartesiano (por defecto, las matrices en computación ponen el índice 0 arriba).


In [ ]:
# julia_mandelbrot.py: Cálculo vectorizado con NumPy
def mandelbrot(xmin=-2.0, xmax=0.5, ymin=-1.25, ymax=1.25, width=800, height=800, max_iter=200):
    """Calcula la matriz para el Conjunto de Mandelbrot (z0 = 0, c variable)."""
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    C = x + 1j * y[:, None]  # Broadcasting: forma (height, width)
    Z = np.zeros_like(C, dtype=np.complex128)
    escape_grid = np.full(C.shape, max_iter, dtype=int)
    mask = np.ones(C.shape, dtype=bool)

    for i in range(max_iter):
        Z[mask] = Z[mask]**2 + C[mask]
        escaped = np.abs(Z) > 2.0
        newly_escaped = escaped & mask
        escape_grid[newly_escaped] = i
        mask &= ~escaped
        if not np.any(mask):
            break

    return escape_grid, (xmin, xmax, ymin, ymax)

def julia(c, xmin=-1.5, xmax=1.5, ymin=-1.5, ymax=1.5, width=800, height=800, max_iter=200):
    """Calcula la matriz para el Conjunto de Julia (c constante, z0 variable)."""
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    Z = x + 1j * y[:, None]
    escape_grid = np.full(Z.shape, max_iter, dtype=int)
    mask = np.ones(Z.shape, dtype=bool)

    for i in range(max_iter):
        Z[mask] = Z[mask]**2 + c
        escaped = np.abs(Z) > 2.0
        newly_escaped = escaped & mask
        escape_grid[newly_escaped] = i
        mask &= ~escaped
        if not np.any(mask):
            break

    return escape_grid, (xmin, xmax, ymin, ymax)

In [ ]:

# Parámetros y ejecución
c_interesante = -0.8 + 0.156j
grid_m, ext_m = mandelbrot(max_iter=300)
grid_j, ext_j = julia(c=c_interesante, max_iter=300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), facecolor="black")

# Mandelbrot
im1 = ax1.imshow(grid_m, extent=ext_m, cmap="twilight_shifted", origin="lower")
ax1.set_title("Conjunto de Mandelbrot", color="white", fontsize=13, fontweight="bold")
ax1.set_xlabel("Re(c)", color="white", fontsize=12)
ax1.set_ylabel("Im(c)", color="white", fontsize=12)
ax1.tick_params(colors="white", labelsize=10)

# Julia
im2 = ax2.imshow(grid_j, extent=ext_j, cmap="twilight_shifted", origin="lower")
ax2.set_title(f"Conjunto de Julia para $c = {c_interesante}$", color="white", fontsize=13, fontweight="bold")
ax2.set_xlabel(r"Re($z_0$)", color="white", fontsize=12)
ax2.set_ylabel(r"Im($z_0$)", color="white", fontsize=12)
ax2.tick_params(colors="white", labelsize=10)

plt.tight_layout()

### 📌 Resumen de Rutinas y Sintaxis Clave

1. **`y[:, None]` o `y[:, np.newaxis]`:**
   Inserta un nuevo eje dimensional al vector 1D de tamaño $(N,)$, transformándolo en una matriz columna de dimensiones $(N, 1)$. Permite que NumPy expanda la memoria mediante *broadcasting* y construya la malla bidimensional $C_{ij} = x_j + i y_i$ sin bucles anidados.

2. **`np.zeros_like(C, dtype=np.complex128)`:**
   Reserva memoria contigua con la forma de $C$ especificando números complejos de doble precisión (64 bits reales + 64 bits imaginarios), evitando pérdidas por redondeo o errores de conversión.

3. **`Z[mask]` (Indexación booleana):**
   Aplica la dinámica únicamente sobre los elementos activos donde `mask == True`. Ahorra órdenes de magnitud de cálculo al ignorar los puntos que ya escaparon al infinito.

4. **`newly_escaped = escaped & mask` y `mask &= ~escaped`:**
   Aísla los puntos que violaron $|z| > 2$ exactamente en la iteración actual para estamparles el valor $i$, e invierte la condición con `~` para apagarlos de la máscara activa.

5. **`ax.imshow(grid, extent, origin="lower", cmap)`:**
   Mapea la matriz discreta de iteraciones a un gradiente de color continuo, orientando $y_{\min}$ en la parte inferior y calibrando las coordenadas cartesianas continuas.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np


# --- FUNCIONES DE CÁLCULO ---
def mandelbrot(
    z0=0.0 + 0.0j,
    xmin=-2.0,
    xmax=0.5,
    ymin=-1.25,
    ymax=1.25,
    width=400,
    height=400,
    max_iter=150,
):
  x = np.linspace(xmin, xmax, width)
  y = np.linspace(ymin, ymax, height)
  C = x + 1j * y[:, None]
  Z = np.full(C.shape, z0, dtype=np.complex128)
  escape_grid = np.full(C.shape, max_iter, dtype=int)
  mask = np.ones(C.shape, dtype=bool)

  for i in range(max_iter):
    Z[mask] = Z[mask] ** 2 + C[mask]
    escaped = np.abs(Z) > 2.0
    newly_escaped = escaped & mask
    escape_grid[newly_escaped] = i
    mask &= ~escaped
    if not np.any(mask):
      break

  return escape_grid, (xmin, xmax, ymin, ymax)


def julia(
    c,
    xmin=-1.5,
    xmax=1.5,
    ymin=-1.5,
    ymax=1.5,
    width=400,
    height=400,
    max_iter=150,
):
  x = np.linspace(xmin, xmax, width)
  y = np.linspace(ymin, ymax, height)
  Z = x + 1j * y[:, None]
  escape_grid = np.full(Z.shape, max_iter, dtype=int)
  mask = np.ones(Z.shape, dtype=bool)

  for i in range(max_iter):
    Z[mask] = Z[mask] ** 2 + c
    escaped = np.abs(Z) > 2.0
    newly_escaped = escaped & mask
    escape_grid[newly_escaped] = i
    mask &= ~escaped
    if not np.any(mask):
      break

  return escape_grid, (xmin, xmax, ymin, ymax)


# --- CONTROLES INTERACTIVOS (IPYWIDGETS) ---
c_real_slider = widgets.FloatSlider(
    value=-0.8,
    min=-2.0,
    max=1.0,
    step=0.005,
    description="Re(c):",
    continuous_update=False,
    layout=widgets.Layout(width="40%"),
)
c_imag_slider = widgets.FloatSlider(
    value=0.156,
    min=-1.5,
    max=1.5,
    step=0.005,
    description="Im(c):",
    continuous_update=False,
    layout=widgets.Layout(width="40%"),
)
z0_real_slider = widgets.FloatSlider(
    value=0.0,
    min=-1.5,
    max=1.5,
    step=0.05,
    description="Re(z0):",
    continuous_update=False,
    layout=widgets.Layout(width="40%"),
)
z0_imag_slider = widgets.FloatSlider(
    value=0.0,
    min=-1.5,
    max=1.5,
    step=0.05,
    description="Im(z0):",
    continuous_update=False,
    layout=widgets.Layout(width="40%"),
)

# Botones predefinidos con casos clásicos interesantes
btn_dendrita = widgets.Button(
    description="Dendrita", button_style="info", layout=widgets.Layout(width="110px")
)
btn_conejo = widgets.Button(
    description="Conejo Douady",
    button_style="info",
    layout=widgets.Layout(width="130px"),
)
btn_san_marco = widgets.Button(
    description="San Marco",
    button_style="info",
    layout=widgets.Layout(width="110px"),
)
btn_galaxia = widgets.Button(
    description="Galaxia", button_style="info", layout=widgets.Layout(width="110px")
)


def set_preset_dendrita(b):
  c_real_slider.value, c_imag_slider.value = 0.0, 1.0


def set_preset_conejo(b):
  c_real_slider.value, c_imag_slider.value = -0.123, 0.745


def set_preset_san_marco(b):
  c_real_slider.value, c_imag_slider.value = -0.75, 0.0


def set_preset_galaxia(b):
  c_real_slider.value, c_imag_slider.value = -0.7, 0.27015


btn_dendrita.on_click(set_preset_dendrita)
btn_conejo.on_click(set_preset_conejo)
btn_san_marco.on_click(set_preset_san_marco)
btn_galaxia.on_click(set_preset_galaxia)

output_plot = widgets.Output()


def actualizar_grafica(c_re, c_im, z0_re, z0_im):
  c_val = complex(c_re, c_im)
  z0_val = complex(z0_re, z0_im)

  # Para interactividad en vivo usamos resolución 400x400 (responde al instante)
  grid_m, ext_m = mandelbrot(
      z0=z0_val, width=400, height=400, max_iter=150
  )
  grid_j, ext_j = julia(
      c=c_val, width=400, height=400, max_iter=150
  )

  with output_plot:
    output_plot.clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5), facecolor="black")

    # Mandelbrot
    ax1.imshow(
        grid_m, extent=ext_m, cmap="twilight_shifted", origin="lower"
    )
    # Marcar el punto c seleccionado sobre el plano de Mandelbrot
    ax1.plot(c_val.real, c_val.imag, "ro", markersize=6, label="Punto $c$ actual")
    ax1.set_title(
        f"Mandelbrot ($z_0 = {z0_val}$)",
        color="white",
        fontsize=12,
        fontweight="bold",
    )
    ax1.set_xlabel("Re(c)", color="white")
    ax1.set_ylabel("Im(c)", color="white")
    ax1.tick_params(colors="white")
    ax1.legend(loc="lower left", fontsize=9)

    # Julia
    ax2.imshow(grid_j, extent=ext_j, cmap="twilight_shifted", origin="lower")
    ax2.set_title(
        f"Julia para $c = {c_val.real:+.3f} {c_val.imag:+.3f}i$",
        color="white",
        fontsize=12,
        fontweight="bold",
    )
    ax2.set_xlabel(r"Re($z_0$)", color="white")
    ax2.set_ylabel(r"Im($z_0$)", color="white")
    ax2.tick_params(colors="white")

    plt.tight_layout()
    plt.show()


# Conectar interacción reactiva
interactive_view = widgets.interactive_output(
    actualizar_grafica,
    {
        "c_re": c_real_slider,
        "c_im": c_imag_slider,
        "z0_re": z0_real_slider,
        "z0_im": z0_imag_slider,
    },
)

# Estructurar la interfaz
fila_botones = widgets.HBox(
    [
        widgets.Label("Valores Notables:"),
        btn_galaxia,
        btn_conejo,
        btn_san_marco,
        btn_dendrita,
    ]
)
fila_c = widgets.HBox([c_real_slider, c_imag_slider])
fila_z0 = widgets.HBox([z0_real_slider, z0_imag_slider])

display(
    widgets.VBox(
        [
            fila_botones,
            widgets.HTML("<b>Parámetro $c$ para Julia:</b>"),
            fila_c,
            widgets.HTML(
                "<b>Condición inicial $z_0$ para Mandelbrot (por defecto 0):</b>"
            ),
            fila_z0,
            output_plot,
        ]
    )
)

---
## Taller de Problemas Propuestos

### Problema 1: El conjunto de Mandelbrot
1. Defina una cuadrícula de números complejos $c = x + iy$ en $-2 \le x \le 2$ y $-2 \le y \le 2$.
2. Para cada punto, aplique la iteración $z_{n+1} = z_n^2 + c$ comenzando con $z_0 = 0$ hasta un máximo de 100 pasos. Si en algún paso $|z| > 2$, registre que ese punto escapó.
3. Grafique el resultado con `plt.imshow` probando primero una resolución pequeña ($100 \times 100$) y luego aumentando a $500 \times 500$.

---

### Problema 2: Los conjuntos de Julia
1. Fije una constante compleja, por ejemplo $c = -0.7 + 0.27015i$ o $c = -0.4 + 0.6i$.
2. Construya la cuadrícula para las condiciones iniciales $z_0 = x + iy$ en $[-1.5, 1.5] \times [-1.2, 1.2]$.
3. Itere hasta 100 veces por punto guardando el conteo de iteraciones y visualice con un mapa de color como `"inferno"` o `"magma"`.

---

### Problema 3: El Juego del Caos y el Triángulo de Sierpinski
1. Defina los tres vértices fijos de un triángulo equilátero: $A = (0, 0)$, $B = (1, 0)$, $C = (0.5, \frac{\sqrt{3}}{2})$.
2. Comience en un punto arbitrario como $(0.5, 0.2)$.
3. En cada paso, elija al azar uno de los 3 vértices con `np.random.default_rng` y muévase al punto medio exacto entre la posición actual y el vértice.
4. Genere $50\,000$ puntos, descarte los primeros 100 pasos transitorios y grafique con `plt.scatter(..., s=0.1)`.

---

### Problema 4: Ramificación Recursiva de un Árbol
1. Escriba una función recursiva `trazar_rama(x, y, longitud, angulo, nivel)` que dibuje un tronco de longitud $L$ con `plt.plot`.
2. En la punta del tronco, bifurque en dos ramas hijas con ángulos $+\theta$ y $-\theta$ (e.g. $\theta \approx 30^\circ$) y factor de reducción de longitud $L_{\text{hija}} = 0.7 L$.
3. Detenga la recursión cuando `nivel == 0` (profundidad de 8 o 9 niveles).
4. *Opcional:* Incorpore una ligera fluctuación aleatoria normal en los ángulos de bifurcación para generar un aspecto biológico realista.

---

### Problema 5: El Mapa Logístico y el Caos Determinista
La ecuación de Robert May para el tamaño poblacional interanual normalizado $x \in [0, 1]$ es:
$$
x_{n+1} = r \, x_n (1 - x_n)
$$
1. Barra 1000 valores de la tasa de reproducción $r \in [2.5, 4.0]$.
2. Para cada $r$, comience en $x_0 = 0.5$ e itere 300 veces sin registrar datos (transitorio hacia el atractor).
3. Itere 100 generaciones adicionales y guarde los valores resultantes de $x$.
4. Grafique los puntos $(r, x)$ mediante `plt.plot(r, x, ',k', alpha=0.25)` para observar la cascada de bifurcación de período hacia el régimen caótico.

---

### Problema 6: Decaimiento Radiactivo con Monte Carlo
1. Inicie con una muestra de $N_0 = 5000$ átomos y probabilidad de decaimiento por intervalo de $p = 0.03$ (con $\Delta t = 1$ s).
2. En cada paso de tiempo, genere $N_{\text{actual}}$ números aleatorios uniformes con `rng.random()`; desintegre a los átomos con valor menor a $p$ y actualice el total.
3. Simule 150 segundos y contraste su curva estocástica frente a la ley analítica $N(t) = N_0 e^{-\lambda t}$ con $\lambda = p$.
4. Estime numéricamente la vida media experimental de la muestra y verifique su coincidencia con $t_{1/2} = \frac{\ln(2)}{p}$.